# 03 — Extract RGB Image Features

## Objective
Create a tabular training dataset from labelled pork packaging images using simple whole-image colour statistics.

## Inputs
- `../data/raw/train/`: training images.
- `../data/raw/train/_annotations.coco.json`: COCO annotations used to derive provisional image-level PASS/FAIL labels.

## Method
1. Reconstruct image labels from defect annotations (same provisional rule used in Notebook 02).
2. Open each training image and convert it to RGB.
3. Calculate the mean and standard deviation of each RGB channel across all pixels.
4. Combine the six numerical features with the filename and PASS/FAIL target.
5. Export the resulting feature table to a CSV file.

## Outputs
- `../data/processed/train_features.csv`, containing `filename`, `mean_red`, `mean_green`, `mean_blue`, `std_red`, `std_green`, `std_blue`, and `label`.

## Interpretation and limitations
These are **global image statistics**: they summarise colour over the whole image rather than detecting or locating individual defects. Lighting, meat/fat appearance, camera position, and packaging can affect them. All labels follow the COCO annotation-based PASS/FAIL assumption.

### Running this review copy

Use the original project `notebooks` folder as the kernel working directory; this copy is stored in its `for review` subfolder. The existing `../data` and `../models` paths depend on that working directory. Run the workflow in order in a fresh kernel. Stored outputs are retained from the source notebook and were not regenerated during this readability review.

## Imports
Load numerical operations (NumPy), tabular tools (pandas), image reading (Pillow), path handling, and COCO JSON parsing.

In [2]:
# Load numerical, image-processing, tabular, and file-handling libraries.
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
import json

## Locate the training images
Define the folder containing the image files. The existing dataset split is already represented by separate `train` and `valid` directories.

In [3]:
# Point to the existing training partition; no new split is created here.
train_folder = Path("../data/raw/train") #setting the path to the training data folder

## Reconstruct PASS/FAIL labels
Read the COCO annotations, resolve category IDs to defect names, and label each training image based on whether at least one annotation is present.

In [4]:
# Read defect annotations attached to the training images.
annotation_path = train_folder / "_annotations.coco.json"

with open(annotation_path, "r") as file:
    coco_data = json.load(file)

# Build category and image lookups for provisional image-level labels.
defect_types = {}
# Create a lookup from category ID to category name
category_names = {}

for category in coco_data["categories"]:
    category_names[category["id"]] = category["name"]
    
# Each annotation identifies an image and a labelled defect category.
    
for annotation in coco_data["annotations"]:
    image_id = annotation["image_id"]
    category_id = annotation["category_id"]
    defect_name = category_names[category_id]

    if image_id not in defect_types:
        defect_types[image_id] = []

    defect_types[image_id].append(defect_name)

image_labels = []

# Create one labelled record for each original image.

for image_record in coco_data["images"]:

    image_id = image_record["id"]
    filename = image_record["file_name"]

    defects = defect_types.get(image_id, [])
    # Retrieve the defects associated with this image, or an empty list if none exist.
    number_of_defects = len(defects)

    if number_of_defects > 0:
        label = "FAIL"
    # An image is marked FAIL when at least one defect annotation is recorded.
    else:
        label = "PASS"

    image_labels.append({
        "image_id": image_id,
        "filename": filename,
        "number_of_defects": number_of_defects,
        "defect_types": defects,
        "label": label
    })

labels_df = pd.DataFrame(image_labels)

## Calculate whole-image RGB features
For each labelled image, calculate six features: the mean and standard deviation of each RGB channel. `axis=(0, 1)` reduces over image height and width, leaving the three colour channels.

In [5]:
# Accumulate a feature dictionary for each image in the training set.
image_features = []

# Iterate over each image and its precomputed label.
for _, row in labels_df.iterrows():

    # Construct the file path using the image filename from the labels table.
    image_path = train_folder / row["filename"]

    # Force a consistent three-channel RGB representation.
    image = Image.open(image_path).convert("RGB")

    # Convert Pillow image data into an array shaped (height, width, 3).
    image_array = np.array(image)

    # Collapse the spatial dimensions to obtain one mean per colour channel.
    mean_rgb = image_array.mean(axis=(0, 1))
    std_rgb = image_array.std(axis=(0, 1))

# Standard deviation measures whole-image variation within each colour channel.

    image_features.append({
        # Keep a filename for traceability and the label for supervised learning.
        "filename": row["filename"],
        "mean_red": mean_rgb[0],
        "mean_green": mean_rgb[1],
        "mean_blue": mean_rgb[2],
        "std_red": std_rgb[0],
        "std_green": std_rgb[1],
        "std_blue": std_rgb[2],
        "label": row["label"]
    })

    features_df = pd.DataFrame(image_features)

# Rebuild the accumulated feature table at this existing location in the loop.
features_df.head()

,filename,mean_red,mean_green,mean_blue,std_red,std_green,std_blue,label
0,frame_with_red_8214_jpg.rf.5acda82172bcf022bd7...,30.301752,28.310728,28.183426,37.372829,28.631737,29.002394,FAIL
1,frame_with_red_1237_jpg.rf.7c920cb9b9d7c993551...,29.648264,30.121911,28.617816,40.769531,29.050063,28.730397,FAIL
2,frame_with_red_6884_jpg.rf.73d4cccf385f78c6473...,29.461821,27.114720,27.660437,40.344334,29.204149,29.422255,FAIL
3,frame_with_red_2737_jpg.rf.8af748cd8904c67c7a0...,49.706422,45.106337,46.502819,41.100140,27.931970,28.625920,PASS
4,frame_with_red_10465_jpg.rf.800ed5e3f0775757ce...,56.781209,52.052703,50.397927,52.871247,40.003867,39.245673,FAIL


## Export the training feature table
Save the dataframe to a CSV for use by the baseline classifier in Notebook 04. This cell assumes `../data/processed/` already exists.

In [6]:
# Define the output location consumed by the subsequent classifier notebook.
output_path = Path("../data/processed/train_features.csv") #output path for the features CSV file

# Export the collected numerical features without a pandas index column.
features_df.to_csv(output_path, index=False)
# Confirm where the file was written and whether it exists.
print("Saved to:", output_path)
print("File exists:", output_path.exists())

Saved to: ..\data\processed\train_features.csv
File exists: True
